In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/datascience-metrology/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Exercise — Module M3: ข้อมูลไม่สะอาด

**วิธีทำ:** เติมช่อง `# TODO` / `...` ให้ทุกเซลล์รันผ่าน แล้วตอบคำถาม markdown ที่กำหนด —
เปิด `book.ipynb` เทียบได้ตลอด (โค้ดชุดเดียวกัน แค่มีคำอธิบายมากกว่า)

**ข้อมูล (citation):** Kohavi & Becker (1996), *Adult*, UCI ML Repository, DOI 10.24432/C5XW20 (CC BY 4.0) ·
LeDoux, J. (2018), *coffee-quality-database* (data © CQI) · กรมควบคุมมลพิษ (2556), pcd.gdcatalog.go.th

In [2]:
import numpy as np  # noqa: F401 — จำเป็นตอนทำ TODO (np.nan)
import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")


DATA = repo_root() / "datasets" / "adult"  # noqa: E305

## ส่วนที่ 1 — ค่าว่างและค่าปลอม (Adult)

In [3]:
# โหลดให้ถูก: ไฟล์นี้ไม่มี header — ตั้งชื่อคอลัมน์เอง 15 ตัว
# (สคีมาอยู่ใน adult.names)
cols = ["age", "workclass", "fnlwgt", "education", "education_num",
        "marital_status", "occupation", "relationship", "race", "sex",
        "capital_gain", "capital_loss", "hours_per_week",
        "native_country", "income"]
adult = pd.read_csv(
    DATA / "adult.data", header=None, names=cols, skipinitialspace=True)
adult.shape


In [4]:
# TODO: ค่าปลอม "?" ทุกคอลัมน์ -> np.nan
# แล้วแสดงจำนวนค่าหาย (เฉพาะคอลัมน์ที่มี)
adult = ...
adult.isna().sum()

SyntaxError: invalid syntax (857345771.py, line 3)

**คำถาม 1.1** คอลัมน์ไหนมีค่าหาย และแต่ละคอลัมน์หายกี่เปอร์เซ็นต์ของ 32,561 แถว?

**คำถาม 1.2** ถ้าเราตัดแถวที่ `workclass` หายทิ้งหมด เราเสียข้อมูลกี่เปอร์เซ็นต์ — คุ้มหรือควรเติม?

In [5]:
# ทางที่ 1 — ตัดแถวที่ workclass หรือ occupation หายทิ้ง
adult_drop = adult.dropna(subset=["workclass", "occupation"])
print(f"เหลือ {len(adult_drop):,} จาก {len(adult):,} แถว")


TypeError: object of type 'ellipsis' has no len()

## ส่วนที่ 2 — สะกดไม่ตรงและหน่วยปน (Coffee)

In [6]:
coffee = pd.read_csv(
    repo_root() / "datasets" / "coffee_quality" / "arabica_data_cleaned.csv")

# TODO: คอลัมน์ Altitude เป็นข้อความ
# (เช่น "1450 msnm", "1200 - 1500")
# สกัดตัวเลขแรกด้วย str.extract ให้ได้คอลัมน์ altitude_m (dtype float)
coffee["altitude_m"] = ...
print("แปลงได้:", coffee["altitude_m"].notna().sum())

แปลงได้: 1311


In [7]:
# รวมพี่น้องที่สะกดเพี้ยนด้วย dictionary mapping
fix_variety = {"V. Colombia": "Colombia", "Tico": "Typica",
               "Bourbon Ponan": "Bourbon"}
coffee["Variety"] = coffee["Variety"].replace(fix_variety)
print("เหลือ", coffee["Variety"].nunique(), "กลุ่ม")


เหลือ 29 กลุ่ม


**คำถาม 2.1** คุณรวมกลุ่มพันธุ์กาแฟด้วยเหตุผลอะไรบ้าง — อันไหนที่คุณ**ไม่แน่ใจ**และจะไม่รวม?

**คำถาม 2.2** คอลัมน์ `altitude_m` แปลงได้ไม่ครบทุกแถว คุณจะบอกผู้อ่านรายงานอย่างไรเรื่องนี้?

## ส่วนที่ 3 — ไฟล์ไทย: Encoding + วันที่แปลก (PCD 2556)

In [8]:
# TODO: ไฟล์นี้ encoding ไม่ใช่ UTF-8 — ลองรันแล้วอ่าน error
# แล้วแก้ด้วย encoding ที่ถูกต้อง (มาตรฐานภาษาไทยก่อนยุค UTF-8)
legacy = pd.read_csv(
    repo_root() / "datasets" / "pcd_pm25_legacy"
    / "aqpm2.5bangkokmetropolitanregion2556.csv",
    encoding=...,   # TODO: รหัสอักขระภาษาไทย
                    # ก่อนยุค UTF-8
)
legacy.head(3)

TypeError: lookup() argument must be str, not ellipsis

In [9]:
# (1) ตัดแถวหมายเหตุท้ายตาราง — แถวที่คอลัมน์แรกไม่ขึ้นต้นด้วยตัวเลข
first_col = legacy.iloc[:, 0].astype("string")
data = legacy[first_col.str.match(r"\d", na=False)].copy()
print("แถวข้อมูล:", len(data))
# TODO: (2) melt เป็นตารางยาว -> คอลัมน์ date_raw, pm25
long = ...
# TODO: (3) แปลง date_raw (รูปแบบ M/D/YYYY) เป็น datetime แบบไม่พังถ้าผิดรูป
long["date"] = ...
print("วันที่ parse ไม่ได้:", long["date"].isna().sum())


TypeError: 'ellipsis' object does not support item assignment

**คำถาม 3.1** ทำไมต้องใช้ `errors="coerce"` แทนที่จะปล่อยให้ pandas parse แบบเข้มงวด —
แบบไหน "ปลอดภัยกว่า" สำหรับการส่งต่อให้เพื่อนร่วมทีมวิเคราะห์ต่อ?

## ส่วนที่ 4 — เช็คลิสต์ของคุณเอง

**คำถาม 4.1 (ตอบเป็นข้อความ)** จาก 6 ข้อของเช็คลิสต์ใน `book.ipynb` §5 —
ข้อมูลชุดไหนในที่ทำงานของคุณ มีปัญหาข้อไหนมากที่สุด เขียน 2-3 บรรทัด